# EXTRACCIÓN DE DATA TMDb

### 1. Importaciones

In [1]:
import sys
import json
import requests
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
import os

### 2. Configuración del proyecto

In [2]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.utils.paths import (
    BRONZE_MOVIELENS_DIR,
    BRONZE_TMDB_DIR
)

In [3]:
print("Proyecto:", PROJECT_ROOT)
print("MovieLens:", BRONZE_MOVIELENS_DIR)
print("TMDb:", BRONZE_TMDB_DIR)

Proyecto: d:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch
MovieLens: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\bronze\movielens
TMDb: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\bronze\tmdb


### 3. Carga de credenciales

In [4]:
load_dotenv(PROJECT_ROOT / ".env")

TMDB_TOKEN = os.getenv("TMDB_TOKEN")

if not TMDB_TOKEN:
    raise ValueError("No se encontró TMDB_TOKEN en el archivo .env")

### 4. Lectura de IDs TMDb desde MovieLens

In [5]:
ruta_movielens = BRONZE_MOVIELENS_DIR / "ml-32m"

links = pd.read_csv(
    ruta_movielens / "links.csv"
)

links.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [6]:
print(links.shape)
print(links.columns.tolist())

(87585, 3)
['movieId', 'imdbId', 'tmdbId']


In [7]:
tmdb_ids = (
    links["tmdbId"]
    .dropna()
    .astype(int)
    .unique()
)

print("IDs TMDb disponibles:", len(tmdb_ids))

IDs TMDb disponibles: 87425


### 5. Prueba de conexión a TMDb

In [8]:
headers = {
    "Authorization": f"Bearer {TMDB_TOKEN}",
    "accept": "application/json"
}

In [9]:
tmdb_id = tmdb_ids[0]

url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

params = {
    "language": "es-ES",
    "append_to_response": "credits,keywords"
}

respuesta = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

respuesta.raise_for_status()

pelicula = respuesta.json()

In [10]:
print(pelicula["id"])
print(pelicula["title"])

862
Toy Story


### 6. Respuesta de prueba en Bronze

In [11]:
archivo_prueba = BRONZE_TMDB_DIR / f"movie_{tmdb_id}.json"

with open(
    archivo_prueba,
    "w",
    encoding="utf-8"
) as archivo:
    json.dump(
        pelicula,
        archivo,
        ensure_ascii=False,
        indent=2
    )

In [12]:
print("Guardado en:", archivo_prueba)

Guardado en: D:\INSTITUTO CONTINENTAL\5. CICLO - 5\PROYECTO PRODUCTIVO IIIA\PP_CineMatch\data\bronze\tmdb\movie_862.json


### 7. Extracción de varias películas

In [13]:
ids_prueba = tmdb_ids[:10]

In [14]:
for tmdb_id in ids_prueba:

    archivo_salida = (
        BRONZE_TMDB_DIR
        / f"movie_{tmdb_id}.json"
    )

    # Si ya existe, no volver a descargar
    if archivo_salida.exists():
        print(f"{tmdb_id} ya existe.")
        continue

    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"

    params = {
        "language": "es-ES",
        "append_to_response": "credits,keywords"
    }

    try:
        respuesta = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        respuesta.raise_for_status()

        datos = respuesta.json()

        with open(
            archivo_salida,
            "w",
            encoding="utf-8"
        ) as archivo:
            json.dump(
                datos,
                archivo,
                ensure_ascii=False,
                indent=2
            )

        print(f"{tmdb_id}: OK")

    except requests.RequestException as error:
        print(f"{tmdb_id}: ERROR - {error}")

862 ya existe.
8844: OK
15602: OK
31357: OK
11862: OK
949: OK
11860: OK
45325: OK
9091: OK
710: OK


### 9. Validación de Bronze

In [15]:
archivos_tmdb = list(
    BRONZE_TMDB_DIR.glob("movie_*.json")
)

print(
    "Archivos TMDb descargados:",
    len(archivos_tmdb)
)

Archivos TMDb descargados: 10


### 10. Validación de tamaño de archivo

In [16]:
for archivo in archivos_tmdb[:10]:

    tamano_kb = archivo.stat().st_size / 1024

    print(
        archivo.name,
        "->",
        round(tamano_kb, 2),
        "KB"
    )

movie_11860.json -> 54.78 KB
movie_11862.json -> 38.57 KB
movie_15602.json -> 30.99 KB
movie_31357.json -> 35.29 KB
movie_45325.json -> 23.96 KB
movie_710.json -> 49.93 KB
movie_862.json -> 101.21 KB
movie_8844.json -> 69.69 KB
movie_9091.json -> 93.24 KB
movie_949.json -> 182.23 KB



### 9. Validación de contenido

In [17]:
with open(
    archivos_tmdb[0],
    "r",
    encoding="utf-8"
) as archivo:
    ejemplo = json.load(archivo)

print(ejemplo.keys())

dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'credits', 'keywords'])
